# Shear Flow Dataset Exploration

This notebook explores the `shear_flow` dataset from the `the_well` benchmark. We will look at the data structure, metadata, and visualize the key physical fields.

In [ ]:
import os
import sys
import torch
import matplotlib.pyplot as plt
import numpy as np
from the_well.data import WellDataModule

# Dataset path on HPC-work
DATA_ROOT = "/home/un212/rds/hpc-work/datasets"
DATASET_NAME = "shear_flow"

print(f"Data Root: {DATA_ROOT}")
print(f"Dataset: {DATASET_NAME}")

## 1. Data Loading

We use the `WellDataModule` to load the dataset. We'll disable normalization for this exploration to see the raw physical values.

In [ ]:
datamodule = WellDataModule(
    DATA_ROOT,
    DATASET_NAME,
    batch_size=1, 
    n_steps_input=10,
    n_steps_output=1,
    use_normalization=False
)

test_loader = datamodule.test_dataloader()
batch = next(iter(test_loader))

print("Batch keys:", batch.keys())
print(f"Input fields shape: {batch['input_fields'].shape}") # (B, T, X, Y, C)
print(f"Output fields shape: {batch['output_fields'].shape}")

## 2. Metadata Inspection

Let's look at the field names and resolutions defined in the metadata.

In [ ]:
metadata = datamodule.train_dataset.metadata
print(f"Spatial Resolution: {metadata.spatial_resolution}")
print(f"Field Names: {metadata.field_names}")
print(f"Constant Names: {metadata.constant_scalar_names}")

## 3. Visualization

We will visualize the `tracer`, `pressure`, and `velocity` fields for a single sample.

In [ ]:
# Extract data from batch
# batch['input_fields'] is (1, 10, 256, 512, 4)
# Channels: 0: tracer, 1: pressure, 2: velocity_x, 3: velocity_y
data = batch['input_fields'][0, -1].numpy() # Last time step of the input sequence

fig, axes = plt.subplots(2, 2, figsize=(18, 10))

# 1. Tracer
im0 = axes[0, 0].imshow(data[:, :, 0], cmap='viridis', origin='lower')
axes[0, 0].set_title("Tracer Concentration")
fig.colorbar(im0, ax=axes[0, 0])

# 2. Pressure
im1 = axes[0, 1].imshow(data[:, :, 1], cmap='magma', origin='lower')
axes[0, 1].set_title("Pressure")
fig.colorbar(im1, ax=axes[0, 1])

# 3. Velocity Magnitude
v_mag = np.sqrt(data[:, :, 2]**2 + data[:, :, 3]**2)
im2 = axes[1, 0].imshow(v_mag, cmap='inferno', origin='lower')
axes[1, 0].set_title("Velocity Magnitude")
fig.colorbar(im2, ax=axes[1, 0])

# 4. Velocity X-Component (Shear Profile)
im3 = axes[1, 1].imshow(data[:, :, 2], cmap='RdBu_r', origin='lower')
axes[1, 1].set_title("Velocity X (Shear)")
fig.colorbar(im3, ax=axes[1, 1])

plt.tight_layout()
plt.show()

## 4. Temporal Evolution

Let's see how the tracer evolves over 10 steps in the input sequence.

In [ ]:
seq = batch['input_fields'][0].numpy() # (10, 256, 512, 4)

fig, axes = plt.subplots(2, 5, figsize=(25, 8))
for t in range(10):
    ax = axes[t // 5, t % 5]
    ax.imshow(seq[t, :, :, 0], cmap='viridis', origin='lower')
    ax.set_title(f"Tracer t={t}")
    ax.axis('off')

plt.suptitle("Temporal Evolution of Tracer Concentration")
plt.tight_layout()
plt.show()